In [2]:
from pathlib import Path
import cv2
from ultralytics import YOLO
import pandas as pd

In [3]:
VIDEO_PATH = "//172.16.222.2/EBVrecordings/2024-TrafficMonitoring/data_collection/week_32/box_3/2024_08_06_16_52_08_recordings/video.avi"
OUTPUT_DIR = Path("week_32/box_3/2024_08_06_16_52_08_recordings/labels")
TIMESTAMP_PATH = "//172.16.222.2/EBVrecordings/2024-TrafficMonitoring/data_collection/week_32/box_3/2024_08_06_16_52_08_recordings/timestamps.csv"
FRAMES_PER_CLIP = 5400
CONF_THRESHOLD = 0.1

In [4]:
df_timestamps = pd.read_csv(TIMESTAMP_PATH)
SHIFT = df_timestamps.index[df_timestamps["event_time"] != 0][0]

In [6]:
model = YOLO("yolo26x.pt")

cap = cv2.VideoCapture(VIDEO_PATH)
global_frame_idx = 0
clips_to_skip = 74
t = 0

while t < SHIFT:
    ret, frame = cap.read()
    t += 1

while global_frame_idx < (FRAMES_PER_CLIP * clips_to_skip):
    ret, frame = cap.read()
    global_frame_idx += 1

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    clip_idx = global_frame_idx // FRAMES_PER_CLIP
    frame_in_clip = global_frame_idx % FRAMES_PER_CLIP

    clip_dir = OUTPUT_DIR / f"clip_{clip_idx:03d}"
    clip_dir.mkdir(parents=True, exist_ok=True)

    if global_frame_idx % FRAMES_PER_CLIP == 0:
        print(f"Processing clip_{clip_idx:03d}")

    label_path = clip_dir / f"{frame_in_clip}.txt"

    results = model(frame, conf=CONF_THRESHOLD, verbose=False)[0]

    with open(label_path, "w") as f:
        for box in results.boxes:
            cls = int(box.cls.item())
            conf = float(box.conf.item())
            x_center, y_center, width, height = box.xywhn[0].tolist()

            f.write(
                f"{cls} "
                f"{x_center:.6f} {y_center:.6f} "
                f"{width:.6f} {height:.6f} "
                f"{conf:.6f}\n"
            )

    global_frame_idx += 1

cap.release()
print("Done.")

KeyboardInterrupt: 

In [5]:
LABEL_DIR = Path("week_32/box_3/2024_08_06_16_52_08_recordings/labels/clip_000")
FRAMES_TO_SHOW = 5400   # first minute
WINDOW_NAME = "YOLO Labels - First Minute"

def load_labels(label_path):
    """
    Loads labels from a file.
    Expected format per line:
    class x_center y_center width height confidence
    """
    detections = []

    if not label_path.exists():
        return detections

    with open(label_path, "r") as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) != 6:
                continue

            cls = int(parts[0])
            x_center = float(parts[1])
            y_center = float(parts[2])
            width = float(parts[3])
            height = float(parts[4])
            conf = float(parts[5])

            detections.append((cls, x_center, y_center, width, height, conf))

    return detections


def draw_detections(frame, detections):
    h, w = frame.shape[:2]

    for cls, x_center, y_center, bw, bh, conf in detections:
        # Convert normalized YOLO coords to pixel coords
        x_center_px = x_center * w
        y_center_px = y_center * h
        bw_px = bw * w
        bh_px = bh * h

        x1 = int(x_center_px - bw_px / 2)
        y1 = int(y_center_px - bh_px / 2)
        x2 = int(x_center_px + bw_px / 2)
        y2 = int(y_center_px + bh_px / 2)

        # Clamp to image bounds
        x1 = max(0, min(x1, w - 1))
        y1 = max(0, min(y1, h - 1))
        x2 = max(0, min(x2, w - 1))
        y2 = max(0, min(y2, h - 1))

        # Draw box
        cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)

        # Draw label text
        text = f"class {cls} | {conf:.2f}"
        text_y = max(20, y1 - 8)
        cv2.putText(
            frame,
            text,
            (x1, text_y),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.6,
            (0, 255, 0),
            2,
            cv2.LINE_AA
        )

    return frame


def main():
    cap = cv2.VideoCapture(VIDEO_PATH)

    if not cap.isOpened():
        print(f"Could not open video: {VIDEO_PATH}")
        return
    
    t = 0

    while t < SHIFT:
        ret, frame = cap.read()
        t += 1

    frame_idx = 0

    while cap.isOpened() and frame_idx < FRAMES_TO_SHOW:
        ret, frame = cap.read()
        if not ret:
            print("Reached end of video or failed to read frame.")
            break

        label_path = LABEL_DIR / f"{frame_idx}.txt"
        detections = load_labels(label_path)

        annotated = draw_detections(frame.copy(), detections)

        # Show frame number too
        cv2.putText(
            annotated,
            f"Frame: {frame_idx}",
            (20, 30),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.8,
            (0, 255, 255),
            2,
            cv2.LINE_AA
        )

        cv2.imshow(WINDOW_NAME, annotated)

        # Controls:
        # q = quit
        # space = pause/unpause
        key = cv2.waitKey(1) & 0xFF

        if key == ord("q"):
            break
        elif key == ord(" "):
            # pause until space pressed again or q
            while True:
                paused_key = cv2.waitKey(0) & 0xFF
                if paused_key == ord(" "):
                    break
                elif paused_key == ord("q"):
                    cap.release()
                    cv2.destroyAllWindows()
                    return

        frame_idx += 1

    cap.release()
    cv2.destroyAllWindows()


if __name__ == "__main__":
    main()